Visualize images and segmentations

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import ipywidgets as widgets
import pandas as pd 
# ============================
# Utility Functions
# ============================

def load_volume(path):
    """Load a 3D volume from NIfTI (.nii.gz), NPZ, or NPY format."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix == ".npy":
        return np.load(path)
    elif path.suffix == ".npz":
        data = np.load(path)
        key = list(data.keys())[0]
        return data[key]
    elif path.suffix in [".nii", ".gz"]:
        return nib.load(str(path)).get_fdata()
    else:
        raise ValueError(f"Unsupported format: {path.suffix}")


def window_ct_hu(ct_hu, level=50, width=350):
    """Window the CT image for visualization."""
    lower, upper = level - width / 2.0, level + width / 2.0
    ct_clipped = np.clip(ct_hu, lower, upper)
    return (ct_clipped - lower) / (upper - lower + 1e-6)


def build_label_overlay(label_slice):
    """Return label colormap for segmentation overlay."""
    unique_labels = np.unique(label_slice)
    if np.array_equal(unique_labels, [0]) or np.array_equal(unique_labels, [0, 1]):
        cmap = ListedColormap([[0,0,0,0], [1,0,0,0.45]])  # red overlay
        return label_slice.astype(np.int32), cmap, (0, 1)
    colors = [[0, 0, 0, 0]]
    rng = np.random.default_rng(42)
    for _ in range(int(unique_labels.max())):
        colors.append([*rng.random(3), 0.45])
    cmap = ListedColormap(colors)
    return label_slice.astype(np.int32), cmap, (0, int(unique_labels.max()))


def get_slice(volume, axis, idx):
    """Extract a 2D slice along the chosen axis."""
    if axis == "axial":
        return volume[:, :, idx]
    elif axis == "coronal":
        return volume[:, idx, :]
    elif axis == "sagittal":
        return volume[idx, :, :]
    else:
        raise ValueError(f"Invalid axis: {axis}. Must be 'axial', 'coronal', or 'sagittal'.")


# ============================
# Main Visualization Function
# ============================

def visualize_case(ct_path, label_path, uid, target, axis="axial", save_dir=None,
                   window_level=50, window_width=350, save_all_slices=False):
    """Visualize a single case interactively, optionally saving overlays."""
    ct = load_volume(ct_path)
    label = load_volume(label_path)
    assert ct.shape == label.shape, f"Shape mismatch for {uid}: {ct.shape} vs {label.shape}"

    axis_to_dim = {"axial": 2, "coronal": 1, "sagittal": 0}
    n_slices = ct.shape[axis_to_dim[axis]]

    out_uid_dir = None
    if save_all_slices and save_dir:
        out_uid_dir = Path(save_dir) / str(uid)
        out_uid_dir.mkdir(parents=True, exist_ok=True)

    def plot_slice(idx):
        ct_slice = get_slice(ct, axis, idx)
        label_slice = get_slice(label, axis, idx)
        ct_img = window_ct_hu(ct_slice, window_level, window_width)
        lab_for_plot, lab_cmap, lab_vminmax = build_label_overlay(label_slice)

        plt.figure(figsize=(6, 6))
        plt.imshow(ct_img.T, cmap="gray", origin="lower")
        plt.imshow(lab_for_plot.T, cmap=lab_cmap, origin="lower",
                   vmin=lab_vminmax[0], vmax=lab_vminmax[1])
        plt.axis("off")
        plt.title(f"UID {uid} | Class: {target} | {axis.capitalize()} slice {idx}/{n_slices}")
        plt.tight_layout()

        if out_uid_dir:
            out_path = out_uid_dir / f"{axis}_slice{idx:03d}.png"
            plt.savefig(out_path, bbox_inches='tight')
            plt.close()
        else:
            plt.show()

    # Interactive slider
    slice_slider = widgets.IntSlider(
        value=n_slices // 2,
        min=0,
        max=n_slices - 1,
        step=1,
        description=f'UID {uid}',
        continuous_update=False
    )

    widgets.interact(plot_slice, idx=slice_slider)

    # Optionally save all slices automatically
    if save_all_slices and out_uid_dir:
        print(f"Saving all slices for UID {uid} → {out_uid_dir}")
        for i in range(n_slices):
            plot_slice(i)


def visualize_dataset(uids, img_root, label_root, axis="axial", save_dir=None, save_all_slices=False):
    """Visualize multiple UIDs."""
    img_root, label_root = Path(img_root), Path(label_root)
    df = pd.read_csv(f"/data/colon_cancer/Classifier/ColonCancer/splits.csv")
    for uid in uids:
        # try naming patterns
        possible_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"colon_{uid:03d}.nii.gz", f"{uid}.npy", f"{uid}.npz"
        ]
        
        
        img_path = next((img_root / n for n in possible_names if (img_root / n).exists()), None)
        label_path = next((label_root / n for n in possible_names if (label_root / n).exists()), None)
        print(img_path, label_path)
        if img_path is None or label_path is None:
            print(f"Skipping UID {uid}: missing image or label")
            continue

        row = df[df["UID"] == uid]
        target = row["target"].values[0]
        if target == 0:
            target = "div"
        else:
            target = "cc"

        visualize_case(img_path, label_path, uid, axis=axis, target=target,
                       save_dir=save_dir, save_all_slices=save_all_slices)


# ============================
# Example usage
# ============================

if __name__ == "__main__":
    uids = [108]
    img_root = "/data/colon_cancer/Classifier/ColonCancer/pp_Tr_npz"
    label_root = "/data/colon_cancer/Classifier/ColonCancer/resampledTr/labels_resampled"
    
    save_dir = "/data/benchaaben//classifier/ct_overlays"

    visualize_dataset(uids, img_root, label_root,
                      axis="axial", save_dir=save_dir, save_all_slices=False)
    


    

Visualize training samples: 

In [21]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import nibabel as nib
import torch 
import numpy as np
import torchvision.transforms.functional as F
import random
import torchio as tio
import os
# ----------------------------
# Load dataset
# ----------------------------
project_root = Path("/data/benchaaben/ColonCancerDetection/classifier")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from data import ColonCancer
from torch.utils.data import DataLoader
patch_shape = (32, 156, 156)

dataset = ColonCancer(dataset_name= "Dataset100_CC", patch_size=patch_shape, split="train", path_root="/data/colon_cancer/CC_Detection",use_labels=True,return_full_image=False)
loader = DataLoader(dataset, batch_size=1, shuffle=True)

sample = next(iter(loader))
print(sample["source"].shape)

image = sample["source"][0,0] # shape [ D, H, W]
label = sample["source"][0,1] # shape [ D, H, W]


subject = tio.Subject(
    image=tio.ScalarImage(tensor=image.unsqueeze(0)),  
    label=tio.LabelMap(tensor=label.unsqueeze(0))
)

transform = tio.Compose([
    tio.RandomFlip(axes=('LR', 'AP', 'IS'), flip_probability=0.4),
    tio.RandomAffine(degrees=5)
])

# Apply transforms
transformed = transform(subject)
transformed_patch = transformed['image'].data.squeeze(0)      
transformed_label = transformed['label'].data.squeeze(0)  


# ----------------------------
# Visualization function
# ----------------------------
def view_transformed(slice_idx):
    fig, axes = plt.subplots(1, 1, figsize=(10,5))
    
    # Original
    axes.imshow(image[slice_idx, :, : ], cmap='gray')
    axes.imshow(label[slice_idx, :, :], cmap='Reds', alpha=0.2)
    axes.set_title(f'Original Slice {slice_idx}')
    axes.axis('off')
    """
    # Apply transform
    axes[1].imshow(transformed_patch[slice_idx, :, :], cmap='gray')
    axes[1].imshow(transformed_label[slice_idx, :, :], cmap='Reds', alpha=0.4)
    axes[1].set_title(f'Transformed Slice {slice_idx}')
    axes[1].axis('off')
    """
    #os.makedirs(f"/data/benchaaben/classifier/{uid}", exist_ok=True)
    #save_path = os.path.join(f"/data/benchaaben/classifier/{uid}/pred_{slice_idx}.png")
    #plt.savefig(save_path, bbox_inches='tight', dpi=150)
    #plt.show() 
d = image.shape[0]
slider = widgets.IntSlider(min=0, max=d-1, step=1, value=d//2, description='Slice')
widgets.interact(view_transformed, slice_idx=slider)


Loaded 16 subjects for Split='train'
torch.Size([1, 2, 32, 156, 156])


interactive(children=(IntSlider(value=16, description='Slice', max=31), Output()), _dom_classes=('widget-inter…

<function __main__.view_transformed(slice_idx)>

Histograms for data shapes analysis: 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

def parse_numeric_tuple_or_list(s):
    """
    Convert a string like '(1.0, 1.0, 1.5)' or '[1.0, 1.0, 1.5]' to a numpy array of floats.
    """
    if isinstance(s, str):
        try:
            t = ast.literal_eval(s)
            return np.array(t, dtype=float)
        except:
            raise ValueError(f"Cannot parse {s} as a numeric tuple/list")
    return np.array(s, dtype=float)

def plot_metric_histogram(csv_path, metric_column="cropped_shape", percentiles=(10, 90)):
    """
    Plots histograms for a given numeric metric column from a CSV.
    Supports columns with tuple/list-like entries (e.g., spacing or shapes) or single numeric values.
    """
    df = pd.read_csv(csv_path)

    # Check if the first value looks like a tuple or list
    sample_val = df[metric_column].iloc[0]
    if isinstance(sample_val, str) and (sample_val.startswith("(") or sample_val.startswith("[")):
        df[metric_column] = df[metric_column].apply(parse_numeric_tuple_or_list)
        values = np.stack(df[metric_column].values)  # shape (N, D)
        axis_labels = [f"dim {i}" for i in range(values.shape[1])]
    else:
        # Single numeric column
        values = df[metric_column].to_numpy(dtype=float).reshape(-1, 1)
        axis_labels = [metric_column]

    # Plot histograms
    fig, axs = plt.subplots(1, values.shape[1], figsize=(5 * values.shape[1], 4))
    if values.shape[1] == 1:
        axs = [axs]

    for i, ax in enumerate(axs):
        vals = values[:, i]
        mean_val = np.mean(vals)
        median_val = np.median(vals)
        min_val = np.min(vals)
        max_val = np.max(vals)
        p_low, p_high = np.percentile(vals, percentiles)

        ax.hist(vals, bins=20, color="lightcoral", edgecolor="black", alpha=0.7)
        ax.axvline(mean_val, color="orange", linestyle="--", label=f"Mean: {mean_val:.2f}")
        ax.axvline(median_val, color="red", linestyle="-.", label=f"Median: {median_val:.2f}")
        ax.axvline(min_val, color="green", linestyle=":", label=f"Min: {min_val:.2f}")
        ax.axvline(max_val, color="purple", linestyle=":", label=f"Max: {max_val:.2f}")
        ax.axvline(p_low, color="gray", linestyle="--", label=f"{percentiles[0]}th: {p_low:.2f}")
        ax.axvline(p_high, color="gray", linestyle="--", label=f"{percentiles[1]}th: {p_high:.2f}")

        ax.set_title(f"{metric_column} - {axis_labels[i]}")
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.suptitle(f"Distribution of {metric_column}", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


plot_metric_histogram("/data/colon_cancer/Classifier/resampledTr/resampling_metadata.csv", metric_column="resampled_size")

#plot_metric_histogram("/data/colon_cancer/Classifier/raw_croppedTs/cropping_metadata.csv", metric_column="cropped_shape")

In [ ]:
###################################### generate shapes statistics from pkl. files generated by the segmentation model #####################################
import pickle
from pathlib import Path
import numpy as np
import json
import pandas as pd

def analyze_pkl_folder(pkl_folder: str, output_json: str, output_csv: str = None):
    """
    Analyze a folder of .pkl files containing dictionaries of properties.
    Computes mean, median, min, max, 10th and 90th percentiles per dimension for each property.
    Optionally saves all individual values to a CSV for histogram plotting.
    """
    pkl_folder = Path(pkl_folder)
    if not pkl_folder.exists():
        raise FileNotFoundError(f"Folder not found: {pkl_folder}")

    # Initialize storage
    properties = {
        "shape_before_cropping": [],
        "shape_after_crop_before_resampling": [],
        "shape_after_resampling": [],
        "spacing": []
    }

    # Iterate over .pkl files
    for pkl_file in pkl_folder.glob("*.pkl"):
        with open(pkl_file, "rb") as f:
            data = pickle.load(f)

        for key in properties.keys():
            if key in data:
                properties[key].append(np.array(data[key]))

    # Helper to compute statistics
    def compute_stats(array_list):
        arr = np.stack(array_list, axis=0)
        stats = {
            "mean_shape": np.mean(arr, axis=0).tolist(),
            "median_shape": np.median(arr, axis=0).tolist(),
            "min_shape": np.min(arr, axis=0).tolist(),
            "max_shape": np.max(arr, axis=0).tolist(),
            "10pct_shape": np.percentile(arr, 10, axis=0).tolist(),
            "90pct_shape": np.percentile(arr, 90, axis=0).tolist()
        }
        return stats

    # Compute stats for each property
    results = {key: compute_stats(values) if values else {} for key, values in properties.items()}

    # Save JSON
    with open(output_json, "w") as f:
        json.dump(results, f, indent=4)
    print(f"Summary statistics saved to {output_json}")

    # Save raw values for histogram plotting if requested
    if output_csv:
        all_data = {}
        for key, values in properties.items():
            all_data[key] = [v.tolist() for v in values]  # convert arrays to lists
        # Convert to a DataFrame for easier plotting
        df = pd.DataFrame({k: pd.Series(v) for k, v in all_data.items()})
        df.to_csv(output_csv, index=False)
        print(f"All individual values saved to {output_csv}")


# Example usage:
# analyze_pkl_folder(
#     pkl_folder="/data/benchaaben/ColonCancerDetection/stats_pkl",
#     output_json="dataset_stats.json",
#     output_csv="dataset_stats_raw.csv"
# )
